In [ ]:
import asyncio
import pandas as pd
import random
import time
import threading
import os
import nest_asyncio
from urllib.parse import quote, urlsplit, urlunsplit, parse_qsl, urlencode
from playwright.async_api import async_playwright
from playwright_stealth import Stealth

nest_asyncio.apply() 

AGENT_ID = 1         # ubah ini untuk setiap agent (misal 0 untuk agent pertama, 1 untuk agent kedua, dst.)
TOTAL_AGENTS = 3      # Ubah ini sesuai jumlah total agent yang kamu gunakan (misal 3 jika ada 3 notebook)

DEFAULT_SEARCH_KEYWORD = "machine learning"
TOPIC_LIST_FILE = "udemy_list_topics.txt"
COMPLETED_TOPICS_FILE = f"udemy_completed_topics_agent{AGENT_ID}.txt"
MIN_COURSES_PER_TOPIC = 900
MAX_COURSES_PER_TOPIC = 1000
RAW_DATA_DIR = "data_mentah"


def load_topics(file_path: str):
    topics = []
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            for raw_line in file:
                line = raw_line.strip()
                if not line:
                    continue

                if ". " in line:
                    prefix, rest = line.split(". ", 1)
                    if prefix.isdigit():
                        line = rest.strip()

                if line:
                    topics.append(line)
    except Exception as e:
        print(f"[!] Gagal membaca file topik '{file_path}': {e}")

    return topics



def load_all_completed_topics(total_agents: int):
    completed_topics = set()
    for i in range(total_agents):
        file_path = f"udemy_completed_topics_agent{i}.txt"
        if os.path.exists(file_path):
            try:
                with open(file_path, "r", encoding="utf-8") as file:
                    for raw_line in file:
                        line = raw_line.strip()
                        if line:
                            completed_topics.add(line)
            except Exception as e:
                print(f"[!] Gagal membaca daftar topik selesai '{file_path}': {e}")
    return completed_topics


def append_completed_topic(file_path: str, topic: str):
    try:
        with open(file_path, "a", encoding="utf-8") as file:
            file.write(f"{topic}\n")
    except Exception as e:
        print(f"[!] Gagal menyimpan status topik selesai '{topic}': {e}")


def build_courses_dataframe(dataset):
    cols = [
        "course_title", "course_id", "lecturer_name", "related_topics", "subject",
        "level", "durations", "ratings", "num_ratings", "student",
        "price", "last_update", "articles", "exercise", "downloadable_resources", "url"
    ]

    df = pd.DataFrame(dataset)
    for col in cols:
        if col not in df.columns:
            df[col] = "N/A"

    return df[cols]


def save_dataset(dataset, filename):
    if not dataset:
        return False

    os.makedirs(RAW_DATA_DIR, exist_ok=True)
    output_path = filename if os.path.isabs(filename) else os.path.join(RAW_DATA_DIR, filename)
    df_courses = build_courses_dataframe(dataset)
    df_courses.to_csv(output_path, index=False, encoding="utf-8")
    print(f"📊 Dataset tersimpan dengan nama: {output_path}")
    return True


def is_valid_scraped_value(value):
    if value is None:
        return False
    text = str(value).strip()
    return text != "" and text.upper() != "N/A"


def sanitize_filename_part(text):
    safe_text = ''.join(character if character.isalnum() or character in (' ', '-', '_') else '_' for character in str(text))
    safe_text = '_'.join(safe_text.strip().split())
    return safe_text.strip('_') or 'untitled_topic'


def normalize_udemy_search_url(url):
    parsed = urlsplit(str(url))
    allowed_keys = {'p', 'q', 'src'}
    filtered_query = [(key, value) for key, value in parse_qsl(parsed.query, keep_blank_values=True) if key in allowed_keys]
    normalized_query = urlencode(filtered_query)
    return urlunsplit((parsed.scheme, parsed.netloc, parsed.path, normalized_query, ''))


def normalize_udemy_course_url(url):
    parsed = urlsplit(str(url).strip())
    if not parsed.scheme or not parsed.netloc:
        return None

    path = parsed.path or ""
    lowered_path = path.lower()
    enroll_index = lowered_path.find("/enroll")
    if enroll_index != -1:
        path = path[:enroll_index]

    path = path.rstrip("/")
    if not path:
        return None

    return urlunsplit((parsed.scheme, parsed.netloc, f"{path}/", "", ""))


def elapsed_seconds_label(start_time):
    return f"{(time.perf_counter() - start_time):.2f}s"


async def main():
    print("=" * 60)
    print(f"🚀 Memulai Crawler Web Udemy - AGENT {AGENT_ID} / {TOTAL_AGENTS}")
    print("=" * 60)
    run_start_time = time.perf_counter()

    search_topics = load_topics(TOPIC_LIST_FILE)
    if not search_topics:
        print(f"[!] Topik dari file tidak ditemukan, fallback ke default: {DEFAULT_SEARCH_KEYWORD}")
        search_topics = [DEFAULT_SEARCH_KEYWORD]


    my_assigned_topics = []
    for idx, topic in enumerate(search_topics):
        if idx % TOTAL_AGENTS == AGENT_ID:
            my_assigned_topics.append(topic)
            
    print(f"[*] Agent {AGENT_ID} mendapat jatah awal {len(my_assigned_topics)} topik berdasarkan index.")

    all_completed_topics = load_all_completed_topics(TOTAL_AGENTS)
    if all_completed_topics:
        print(f"[*] Topik yang sudah selesai oleh SEMUA Agent sebelumnya: {len(all_completed_topics)}")

    search_topics = [topic for topic in my_assigned_topics if topic not in all_completed_topics]
    
    print(f"[*] Agent {AGENT_ID} kebagian {len(search_topics)} topik sisa untuk diproses saat ini.")
    if not search_topics:
        print(f"[✓] Semua topik jatah Agent {AGENT_ID} sudah selesai. Tidak ada tugas tersisa.")
        return

    print(f"[*] Range target course per topik: {MIN_COURSES_PER_TOPIC} - {MAX_COURSES_PER_TOPIC}")

    final_dataset = []
    success_scrape_logs = []
    failed_scrape_logs = []

    async with Stealth().use_async(async_playwright()) as p:
        for topic_index, current_subject in enumerate(search_topics, 1):
            print("\n" + "-" * 60)
            print(f"[*] AGENT {AGENT_ID} - TOPIK {topic_index}/{len(search_topics)}: {current_subject}")
            print("-" * 60)
            print(f"[*] Inisialisasi sesi baru untuk topik '{current_subject}' (restart-like).")

            launch_options = {"headless": True}
            
            browser = await p.chromium.launch(**launch_options)
            
            context = await browser.new_context(
                viewport={"width": 1920, "height": 1080},
                user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
            )
            page = await context.new_page()
            topic_dataset = []

            target_courses_for_topic = random.randint(MIN_COURSES_PER_TOPIC, MAX_COURSES_PER_TOPIC)
            print(f"[*] Target course untuk topik ini: {target_courses_for_topic}")

            course_urls = []
            course_level_by_url = {}
            course_num_ratings_by_url = {}

            encoded_keyword = quote(current_subject)
            search_url = f"https://www.udemy.com/courses/search/?q={encoded_keyword}"

            print("[*] FASE 1: Memanen URL dari Halaman Pencarian...")
            try:
                next_page_url = search_url
                visited_search_pages = set()
                seen_course_urls = set()
                page_number = 1

                while next_page_url and len(course_urls) < target_courses_for_topic:
                    if next_page_url in visited_search_pages:
                        print("    [!] Pagination berputar ke URL yang sama, proses dihentikan.")
                        break

                    visited_search_pages.add(next_page_url)
                    print(f"    [*] Crawl search page {page_number}: {next_page_url}")

                    await page.goto(next_page_url, wait_until="domcontentloaded", timeout=60000)
                    await page.wait_for_timeout(4000)

                    for _ in range(4):
                        await page.evaluate("window.scrollBy(0, 1500)")
                        await page.wait_for_timeout(1000)

                    page_result = await page.evaluate(r"""
                        () => {
                            const allowedLevels = new Set(['all levels', 'beginner', 'intermediate', 'expert']);
                            const cards = [];
                            const searchRoot = document.querySelector('main') || document.body;
                            const courseLinks = Array.from(searchRoot.querySelectorAll('a[href^="/course/"]'));

                            for (const link of courseLinks) {
                                const url = link.href;
                                if (!url || /\/reviews\//i.test(url) || /\/learn\//i.test(url) || /\/enroll(?:\/|\?|$)/i.test(url)) continue;

                                let level = 'N/A';
                                let numRatings = 'N/A';
                                let container = link.closest('article') || link.closest('li') || link.closest('div[class*="vertical-card"]') || link.closest('div');

                                for (let depth = 0; depth < 4 && container; depth += 1) {
                                    const listItems = container.querySelectorAll('li');
                                    for (const item of listItems) {
                                        const text = item.innerText.trim();
                                        const lowerText = text.toLowerCase();

                                        if (level === 'N/A' && allowedLevels.has(lowerText)) {
                                            level = text;
                                        }

                                        if (numRatings === 'N/A') {
                                            const ratingMatch = text.match(/([\d.,]+)\s*ratings?/i);
                                            if (ratingMatch) {
                                                numRatings = ratingMatch[1].replace(/,/g, '');
                                            }
                                        }
                                    }
                                    if (level !== 'N/A' && numRatings !== 'N/A') break;
                                    container = container.parentElement;
                                }

                                if (numRatings === 'N/A' && container) {
                                    const containerText = container.innerText || '';
                                    const ratingMatch = containerText.match(/([\d.,]+)\s*ratings?/i);
                                    if (ratingMatch) {
                                        numRatings = ratingMatch[1].replace(/,/g, '');
                                    }
                                }

                                cards.push({ url, level, num_ratings: numRatings });
                            }

                            const uniqueCards = [];
                            const seen = new Map();
                            for (const card of cards) {
                                const current = seen.get(card.url);
                                if (!current) {
                                    seen.set(card.url, card);
                                    continue;
                                }

                                if (current.level === 'N/A' && card.level !== 'N/A') current.level = card.level;
                                if (current.num_ratings === 'N/A' && card.num_ratings !== 'N/A') current.num_ratings = card.num_ratings;
                            }

                            for (const card of seen.values()) {
                                uniqueCards.push(card);
                            }

                            const paginationNav = document.querySelector('nav.pagination-module--container---ftIz[aria-label="Pagination"]');
                            let nextPageHref = null;

                            if (paginationNav) {
                                const pageLinks = Array.from(paginationNav.querySelectorAll('a[href*="/courses/search/?"]'));
                                const currentUrl = window.location.href;

                                for (const anchor of pageLinks) {
                                    const href = anchor.getAttribute('href');
                                    if (!href) continue;

                                    const rel = (anchor.getAttribute('rel') || '').toLowerCase();
                                    const ariaLabel = (anchor.getAttribute('aria-label') || '').toLowerCase();
                                    const text = (anchor.innerText || '').trim().toLowerCase();
                                    const absoluteHref = new URL(href, window.location.origin).href;

                                    if (absoluteHref === currentUrl) continue;
                                    if (rel === 'next' || ariaLabel.includes('next') || text === 'next') {
                                        nextPageHref = href;
                                        break;
                                    }
                                }

                                if (!nextPageHref && pageLinks.length > 0) {
                                    const currentPageParam = new URL(currentUrl).searchParams.get('p') || '1';
                                    const currentPage = parseInt(currentPageParam, 10) || 1;

                                    const candidateLinks = pageLinks
                                        .map(anchor => anchor.getAttribute('href'))
                                        .filter(Boolean)
                                        .map(href => ({
                                            href,
                                            page: parseInt(new URL(href, window.location.origin).searchParams.get('p') || '1', 10) || 1
                                        }))
                                        .filter(item => item.page > currentPage)
                                        .sort((a, b) => a.page - b.page);

                                    if (candidateLinks.length > 0) {
                                        nextPageHref = candidateLinks[0].href;
                                    }
                                }
                            }

                            return { cards: uniqueCards, next_page_href: nextPageHref };
                        }
                    """)

                    harvested_cards = page_result.get("cards", [])
                    new_urls_in_page = 0

                    for card in harvested_cards:
                        raw_url = card.get("url")
                        url = normalize_udemy_course_url(raw_url)
                        level = card.get("level", "N/A")
                        num_ratings = card.get("num_ratings", "N/A")
                        if not url:
                            continue

                        if "/enroll/" in url.lower():
                            continue

                        if url not in seen_course_urls:
                            seen_course_urls.add(url)
                            course_urls.append(url)
                            new_urls_in_page += 1
                            if level and level != "N/A":
                                course_level_by_url[url] = level
                            if num_ratings and num_ratings != "N/A":
                                course_num_ratings_by_url[url] = num_ratings

                    print(f"        [+] URL baru dari page {page_number}: {new_urls_in_page}")

                    next_href = page_result.get("next_page_href")
                    if not next_href:
                        break

                    next_page_url = normalize_udemy_search_url(f"https://www.udemy.com{next_href}" if next_href.startswith("/") else next_href)
                    page_number += 1

                print(f"    [+] Total URL unik ditemukan untuk topik ini: {len(course_urls)}")

            except Exception as e:
                print(f"    [!] Error FASE 1 untuk topik '{current_subject}': {e}")
                failed_scrape_logs.append({
                    "subject": current_subject,
                    "url": search_url,
                    "stage": "search_page",
                    "error": str(e)
                })
                course_urls = []

            print("\n[*] FASE 2: Membedah Full Data Blueprint...")

            urls_to_visit = course_urls[:target_courses_for_topic]

            for index, url in enumerate(urls_to_visit, 1):
                print(f"    [{index}/{len(urls_to_visit)}] Ekstrak Full Data: {url.split('udemy.com')[1]}")

                try:
                    await page.goto(url, wait_until="domcontentloaded", timeout=45000)
                    await page.wait_for_timeout(3000)
                    try:
                        await page.wait_for_selector('h1', timeout=15000)
                        await page.wait_for_selector('[data-purpose="course-price-text"], [data-purpose="rating-number"]', timeout=15000)
                    except Exception:
                        pass

                    course_data = await page.evaluate(r"""
                        () => {
                            let data = {
                                course_title: document.querySelector('h1') ? document.querySelector('h1').innerText.trim() : "N/A",
                                course_id: "N/A",
                                lecturer_name: "N/A",
                                related_topics: "N/A",
                                level: "N/A",
                                durations: "N/A",
                                ratings: "N/A",
                                num_ratings: "N/A",
                                price: "N/A",
                                last_update: "N/A",
                                articles: "0",
                                exercise: "0",
                                downloadable_resources: "0",
                                student: "N/A"
                            };

                            let bodyCourseId = document.body ? document.body.getAttribute('data-clp-course-id') : null;
                            if (bodyCourseId && /^\d+$/.test(bodyCourseId)) {
                                data.course_id = bodyCourseId;
                            }

                            let instructorSection = document.querySelector('div[class*="instructor-section"]');
                            if (instructorSection) {
                                let instructorLinks = instructorSection.querySelectorAll('a[data-position]');
                                if (instructorLinks.length > 0) {
                                    let instructorList = Array.from(instructorLinks).map(link => ({
                                        name: link.innerText.trim(),
                                        position: parseInt(link.getAttribute('data-position')) || 0
                                    }));
                                    instructorList.sort((a, b) => a.position - b.position);
                                    data.lecturer_name = instructorList.map(inst => inst.name).join(', ');
                                }
                            }
                            if (data.lecturer_name === "N/A") {
                                let lecturerEl = document.querySelector('.instructor-links--names--7UPaw') || document.querySelector('[data-purpose="instructor-name-top"]');
                                if (lecturerEl) data.lecturer_name = lecturerEl.innerText.trim();
                            }

                            let relatedTopicsH2 = document.querySelector('h2[class*="related-topics"]');
                            if (relatedTopicsH2) {
                                let relatedTopicsContainer = relatedTopicsH2.closest('div');
                                if (relatedTopicsContainer) {
                                    let topicsUl = relatedTopicsContainer.querySelector('ul[aria-label="Explore related topics"]');
                                    if (topicsUl) {
                                        let topicLis = topicsUl.querySelectorAll('li');
                                        if (topicLis.length > 0) {
                                            data.related_topics = Array.from(topicLis).map(li => li.innerText.trim()).join(', ');
                                        }
                                    }
                                }
                            }
                            if (data.related_topics === "N/A") {
                                let topicEls = document.querySelectorAll('.topic-menu a, [data-purpose="course-topic-labels"] a');
                                if (topicEls.length > 0) {
                                    data.related_topics = Array.from(topicEls).map(el => el.innerText.trim()).join(', ');
                                }
                            }

                            let lastUpdatedBox = document.querySelector('div.last-updated-module-scss-module__ByfpyW__last-updated.ud-text-sm');
                            if (lastUpdatedBox) {
                                let lastUpdatedSpan = lastUpdatedBox.querySelector('span');
                                if (lastUpdatedSpan) {
                                    data.last_update = lastUpdatedSpan.innerText.replace(/Last updated/i, '').replace(/Terakhir diperbarui/i, '').trim();
                                }
                            }
                            if (data.last_update === "N/A") {
                                let updateEl = document.querySelector('[data-purpose="last-update-date"]');
                                if (updateEl) {
                                    data.last_update = updateEl.innerText.replace(/Last updated/i, '').replace(/Terakhir diperbarui/i, '').trim();
                                }
                            }

                            let priceContainer = document.querySelector('[data-purpose="course-price-text"]');
                            if (priceContainer) {
                                let priceSpans = priceContainer.querySelectorAll('span:not(.ud-sr-only)');
                                if (priceSpans.length > 0) data.price = priceSpans[0].innerText.trim();
                                else data.price = priceContainer.innerText.split('\n')[0];
                            }

                            let ratingEl = document.querySelector('[data-purpose="rating-number"]');
                            if (ratingEl) data.ratings = ratingEl.innerText.trim();

                            let reviewEl = document.querySelector('a[data-purpose="rating"]');
                            if (reviewEl) data.num_ratings = reviewEl.innerText.replace(/[^0-9]/g, '');

                            let incentiveSpans = document.querySelectorAll('ul[class*="incentives-list"] span');
                            incentiveSpans.forEach(span => {
                                let text = span.innerText.toLowerCase();

                                if (text.includes('on-demand video') || text.includes('video')) {
                                    data.durations = text.replace('on-demand video', '').replace('video', '').trim();
                                }
                                if (text.includes('article') || text.includes('artikel')) {
                                    data.articles = text.replace(/[^0-9]/g, '');
                                }
                                if (text.includes('downloadable') || text.includes('sumber')) {
                                    data.downloadable_resources = text.replace(/[^0-9]/g, '');
                                }
                                if (text.includes('exercise') || text.includes('latihan')) {
                                    data.exercise = text.replace(/[^0-9]/g, '');
                                }
                            });

                            let nextData = document.getElementById('__NEXT_DATA__');
                            if (nextData) {
                                try {
                                    let ssrJson = JSON.parse(nextData.innerText);
                                    let searchJsonKeys = (obj) => {
                                        if (!obj || typeof obj !== 'object') return;

                                        if (obj.id !== undefined && typeof obj.id === 'number' && data.course_id === "N/A") data.course_id = String(obj.id);
                                        if (obj.instructional_level_simple !== undefined) data.level = obj.instructional_level_simple;
                                        if (obj.enrollment_count !== undefined) data.student = obj.enrollment_count;
                                        else if (obj.num_learners !== undefined) data.student = obj.num_learners;

                                        Object.values(obj).forEach(searchJsonKeys);
                                    };
                                    searchJsonKeys(ssrJson);
                                } catch (e) {}
                            }

                            if (data.level === "N/A") {
                                let levelEls = document.querySelectorAll('div[data-purpose="instructional-level"]');
                                if (levelEls.length > 0) data.level = levelEls[0].innerText.trim();
                            }

                            if (data.student === "N/A" || data.student === "0") {
                                let stickyHeader = document.querySelector('div.sticky-header-module-scss-module__1owZmq__sticky-header-container');
                                if (stickyHeader) {
                                    let studentCountBox = stickyHeader.querySelector('div.enrollment-count-module-scss-module__3KI-AG__student-count.ud-text-sm');
                                    if (studentCountBox) {
                                        let studentSpan = studentCountBox.querySelector('span');
                                        if (studentSpan) {
                                            let studentText = studentSpan.innerText || studentSpan.textContent || '';
                                            let studentMatch = studentText.replace(/[^0-9]/g, '');
                                            if (studentMatch) data.student = studentMatch;
                                        }
                                    }
                                }
                            }

                            return data;
                        }
                    """)

                    course_data["url"] = url
                    course_data["subject"] = current_subject
                    course_data["level"] = course_level_by_url.get(url, course_data.get("level", "N/A"))
                    course_data["num_ratings"] = course_num_ratings_by_url.get(url, course_data.get("num_ratings", "N/A"))
                    if course_data.get("course_title") in ("N/A", "", None):
                        try:
                            page_title = await page.title()
                            if page_title:
                                course_data["course_title"] = page_title.replace(" | Udemy", "").strip()
                        except Exception:
                            pass

                    final_dataset.append(course_data)
                    topic_dataset.append(course_data)
                    success_log = {
                        "subject": current_subject,
                        "url": url,
                        "course_id": course_data.get("course_id", "N/A"),
                        "course_title": course_data.get("course_title", "N/A"),
                        "status": "success"
                    }

                    field_names = [
                        "course_title", "course_id", "lecturer_name", "related_topics", "subject",
                        "level", "durations", "ratings", "num_ratings", "student",
                        "price", "last_update", "articles", "exercise", "downloadable_resources", "url"
                    ]

                    for field_name in field_names:
                        field_value = course_data.get(field_name, "N/A")
                        if field_name == "subject":
                            field_value = current_subject
                        elif field_name == "url":
                            field_value = url
                        success_log[f"has_{field_name}"] = is_valid_scraped_value(field_value)
                    success_scrape_logs.append(success_log)
                    first_line_fields = field_names[:8]
                    second_line_fields = field_names[8:]
                    status_parts = [
                        f"{field_name} : {'true' if success_log[f'has_{field_name}'] else 'false'}"
                        for field_name in first_line_fields
                    ]
                    print(f"        [✅][{elapsed_seconds_label(run_start_time)}][LOG] {' | '.join(status_parts)}")
                    if second_line_fields:
                        second_status_parts = [
                            f"{field_name} : {'true' if success_log[f'has_{field_name}'] else 'false'}"
                            for field_name in second_line_fields
                        ]
                        print(f"               {' | '.join(second_status_parts)}")

                except Exception as e:
                    failed_log = {
                        "subject": current_subject,
                        "url": url,
                        "stage": "course_detail",
                        "error": str(e)
                    }
                    failed_scrape_logs.append(failed_log)
                    print(
                        f"        [❌][{elapsed_seconds_label(run_start_time)}][LOG] subject={failed_log['subject']} | "
                        f"url={failed_log['url']} | error={failed_log['error']}"
                    )
                    continue

            checkpoint_topic_name = sanitize_filename_part(current_subject)
            checkpoint_filename = f"udemy_dataset_course_{checkpoint_topic_name}.csv"

            if topic_dataset:
                print(f"\n[*][{elapsed_seconds_label(run_start_time)}] Simpan CSV topik: {current_subject}")
                is_saved = save_dataset(topic_dataset, checkpoint_filename)
                if is_saved:
                    append_completed_topic(COMPLETED_TOPICS_FILE, current_subject)
                    print(f"[✓] Topik '{current_subject}' ditandai selesai di {COMPLETED_TOPICS_FILE}")
            else:
                print(f"[!] Tidak ada data valid untuk topik '{current_subject}', file per topik tidak dibuat.")

            await context.close()
            await browser.close()
            print(f"[*] Sesi topik '{current_subject}' ditutup. Lanjut topik berikutnya dengan sesi baru.")

    if not final_dataset:
        print(f"\n[❌] SCRAPING GAGAL UNTUK AGENT.")
        return

    filename = f"udemy_dataset_course_{current_subject}.csv"
    save_dataset(final_dataset, filename)

    print(f"\nDATA BERHASIL DIAMBIL OLEH AGENT {AGENT_ID}!")
    print(f"[*] Total data sukses: {len(success_scrape_logs)}")
    print(f"[*] Total data gagal: {len(failed_scrape_logs)}")

    df_courses = build_courses_dataframe(final_dataset)

    print("\nPreview Kolom (Melihat Data Spesifik Permintaan Anda):")
    preview_cols = ["subject", "course_id", "lecturer_name", "related_topics", "exercise", "downloadable_resources"]
    print(df_courses[preview_cols].head())


def run_in_new_loop():
    loop = asyncio.ProactorEventLoop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(main())


if __name__ == "__main__":
    thread = threading.Thread(target=run_in_new_loop)
    thread.start()
    thread.join()